In [1]:
!wget https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz

--2026-03-17 10:50:33--  https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/87156914/0b363e00-0126-11e9-9e3c-e8c235463bd6?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-17T11%3A49%3A15Z&rscd=attachment%3B+filename%3Dlenta-ru-news.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-17T10%3A49%3A08Z&ske=2026-03-17T11%3A49%3A15Z&sks=b&skv=2018-11-09&sig=X7BCyg8iADQCEsnu4XrUlN7sylzRi8cWFZa4bmxRBJU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mzc0ODIzMywibmJmIjoxNzczNzQ0NjMzLCJwYXRoIjoicmVsZWFzZWFzc2V0cH

In [2]:
!pip install corus
!python3 -m spacy download ru_core_news_sm
!pip install gensim
!pip install razdel
!pip install navec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 64.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 17.7 MB/s eta 0:00:00


In [3]:
import gensim.models
from corus import load_lenta
import spacy
import string
from tqdm import tqdm
import string
import pandas as pd
from razdel import tokenize, sentenize
import pymorphy3
import numpy as np
import re
import random

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve
    )
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer

from functools import lru_cache
from sklearn.base import clone

from tempfile import mkdtemp
from joblib import Memory

import urllib.request
from navec import Navec

In [4]:
from typing_extensions import dataclass_transform
path = 'lenta-ru-news.csv.gz'
records = load_lenta(path)
data_sup = []

for i, record in enumerate(tqdm(records, total=100000)):
    if i >= 100000:
        break
    data_sup.append({
        'text': record.text,
        'title': record.title,
        'topic': record.topic
    })

data = pd.DataFrame(data_sup)

print(data.info())
data.head()

100%|██████████| 100000/100000 [00:12<00:00, 8230.99it/s]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 3 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   text    100000 non-null  object
 1   title   100000 non-null  object
 2   topic   100000 non-null  object
dtypes: object(3)
memory usage: 2.3+ MB
None


,text,title,topic
0,Вице-премьер по социальным вопросам Татьяна Го...,Названы регионы России с самой высокой смертно...,Россия
1,Австрийские правоохранительные органы не предс...,Австрия не представила доказательств вины росс...,Спорт
2,Сотрудники социальной сети Instagram проанализ...,Обнаружено самое счастливое место на планете,Путешествия
3,С начала расследования российского вмешательст...,В США раскрыли сумму расходов на расследование...,Мир
4,Хакерская группировка Anonymous опубликовала н...,Хакеры рассказали о планах Великобритании зами...,Мир


In [5]:
RANDOM_STATE = 42

Проведем такую же предобработку датасета, убрав из него малопредставленные категории

In [6]:
index_to_drop = data[(data['topic'] == 'Оружие') | (data['topic'] == '')].index
data = data.drop(index_to_drop)

Используем методы нормализации данных из ДЗ№1

In [7]:
morph = pymorphy3.MorphAnalyzer()

def normalize_text(s):
    if isinstance(s, np.ndarray):
        if s.dtype.type is np.str_ or s.dtype.type is np.object_:
            s = s.item() if s.size == 1 else str(s)
        else:
            s = str(s)
    s = s.lower()
    return s

@lru_cache(maxsize=100_000)
def lemma_token_cached(token):
      return morph.parse(token)[0].normal_form

def preprocess_one_cached(s, tfidf=False):
    s = normalize_text(s)
    tokens = [t.text for t in tokenize(s)]
    lemmas = [lemma_token_cached(t) for t in tokens]
    if tfidf:
      return ' '.join(lemmas)
    return lemmas

def preprocess_corpus(texts):
    out = []
    for t in tqdm(texts):
      out.append(preprocess_one_cached(str(t)))
    return np.asarray(out, dtype=object)

In [8]:
X = (data["text"] + " " + data["title"] + " " + data["title"]).values
y = data["topic"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=RANDOM_STATE
)

In [9]:
X_train_clear = preprocess_corpus(X_train)

100%|██████████| 59988/59988 [03:11<00:00, 313.51it/s]


Далее обучим w2v модель на заданном трейн сете, используем вектор сайз равный 256, так как далее будут использована модели с размерность равной 300, думаю, сравнение методов при похожем значении такого важного гиперпараметра будет более справедливым. Размерности 200-300 обычно достаточно для захвата семантики Также добавлю, что так как модель будет обучена на CPU, мы не можем себе позволить сильно увеличивать данный параметр. Размер окна = 7 также является компромисом для новостных текстов, чтобы улавливать семантику и не размывать контекст. Парметр минимального количества токенов подобран изходя из рамера датасета, на размер порядка 50к-100к документов 10 - цифра вполне оптимальна

In [10]:
%%time

model = gensim.models.Word2Vec(
    sentences=X_train_clear,
    vector_size=256,
    window=7,
    min_count=10,
    sg=1,
    negative=5,
    epochs=25,
    seed=2023,
)

CPU times: user 1h 24min 22s, sys: 9.65 s, total: 1h 24min 32s
Wall time: 48min 41s


In [18]:
model.wv.most_similar(positive=['правитель'], topn=5)

[('компартия', 0.3936096131801605),
 ('бюст', 0.38983893394470215),
 ('рамат-гана', 0.3805498778820038),
 ('воеводин', 0.3767487704753876),
 ('иудейский', 0.36474862694740295)]

In [20]:
model.wv.most_similar(positive=['спартак'], topn=5)

[('красно-белый', 0.7300438284873962),
 ('цска', 0.6647362112998962),
 ('зенит', 0.6632763743400574),
 ('каррер', 0.6468613147735596),
 ('карреры', 0.6199871301651001)]

In [21]:
model.wv.most_similar(positive=['группировка'], topn=5)

[('террористический', 0.7203096747398376),
 ('иго', 0.6871956586837769),
 ('исламский', 0.6755338311195374),
 ('ахрар', 0.6731649041175842),
 ('аш-ша', 0.6399115324020386)]

In [24]:
model.wv.doesnt_match(['группировка', 'объединение', 'молоко', 'союз', 'альянс'])

'молоко'

In [27]:
model.wv.doesnt_match(['политика', 'партия', 'правительство', 'спорт', 'конституция'])

'спорт'

На визуальном уровне можно сказать, что качество получившихся векторов вполне неплохое

In [9]:
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar

--2026-03-17 10:52:15--  https://storage.yandexcloud.net/natasha-navec/packs/navec_news_v1_1B_250K_300d_100q.tar
Resolving storage.yandexcloud.net (storage.yandexcloud.net)... 213.180.193.243, 2a02:6b8::1d9
Connecting to storage.yandexcloud.net (storage.yandexcloud.net)|213.180.193.243|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26634240 (25M) [application/x-tar]
Saving to: ‘navec_news_v1_1B_250K_300d_100q.tar’

navec_news_v1_1B_25 100%[===================>]  25.40M  10.3MB/s    in 2.5s    

2026-03-17 10:52:18 (10.3 MB/s) - ‘navec_news_v1_1B_250K_300d_100q.tar’ saved [26634240/26634240]



In [10]:
navec_path = 'navec_news_v1_1B_250K_300d_100q.tar'
navec = Navec.load(navec_path)

In [11]:
urllib.request.urlretrieve(
    "https://rusvectores.org/static/models/rusvectores4/ruwikiruscorpora/ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz",
    "ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz"
)

('ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz',
 <http.client.HTTPMessage at 0x7f0e1cc7a300>)

In [12]:
rusvectors_path = 'ruwikiruscorpora_upos_skipgram_300_2_2018.vec.gz'
rusvectors = gensim.models.KeyedVectors.load_word2vec_format(rusvectors_path)

Напишем сразу все методы для векторизации разлияными векторными моделями, так как они имеют слегка разный интерфейм. Далее объединим нормализацию данных перевод, перевод в вектор и fit/eval модели в два метода с интерфейсом для всех векторизаторов

In [13]:
def text_to_vector_navec(text, navec_model):

    word_vectors = []
    for token in text:
      if token in navec_model:
        word_vectors.append(navec_model[token])
      else:
        pass

    if len(word_vectors) == 0:
        return np.zeros(navec_model.dim)

    sentence_vector = np.mean(word_vectors, axis=0)
    return sentence_vector


def text_to_vector_rusvectores(text, model):
    word_vectors = []

    for token in text:
        clean_token = token.lower().strip()
        found = False

        if hasattr(model, 'key_to_index'):
            variants = []

            if clean_token.endswith('ть') or clean_token.endswith('ти'):
                variants = [f"{clean_token}_VERB", f"{clean_token}_INFN"]
            elif clean_token.endswith('ся'):
                variants = [f"{clean_token}_VERB", f"{clean_token}_INFN"]
            elif clean_token.endswith('ый') or clean_token.endswith('ий') or clean_token.endswith('ой'):
                variants = [f"{clean_token}_ADJ"]
            else:
                variants = [
                    f"{clean_token}_NOUN",
                    f"{clean_token}_ADJ",
                    f"{clean_token}_VERB",
                    f"{clean_token}_ADV",
                    f"{clean_token}_PROPN"
                ]

            variants.append(clean_token)

            # Пробуем все варианты
            for variant in variants:
                if variant in model.key_to_index:
                    word_vectors.append(model[variant])
                    found = True
                    break

            if not found:
                pass

    if len(word_vectors) == 0:
        dim = model.vector_size if hasattr(model, 'vector_size') else 300
        return np.zeros(dim)

    sentence_vector = np.mean(word_vectors, axis=0)
    count_ratio = len(word_vectors)/len(text)

    return sentence_vector, count_ratio



def text_to_vector_gensim(text, w2v_model):
    word_vectors = []
    for token in text:
        if token in w2v_model.wv:
            word_vectors.append(w2v_model.wv[token])

    if len(word_vectors) == 0:
        return np.zeros(w2v_model.wv.vector_size)

    return np.mean(word_vectors, axis=0)



def fit_log_reg(X, y, vectorizer, clf, model_type):
  print('Предобработка и нормализация для обучения')
  X_train_clear = preprocess_corpus(X)

  X_train_vectorized = []
  print('Векторизация текста')

  if model_type == 'navec':
    for elem in tqdm(X_train_clear):
      X_train_vectorized.append(text_to_vector_navec(elem, vectorizer))

  elif model_type == 'rusvectors':
    count_ratio = []
    for elem in tqdm(X_train_clear):
      result = text_to_vector_rusvectores(elem, vectorizer)
      X_train_vectorized.append(result[0])
      count_ratio.append(result[1])
    print(f'Удалось найти {sum(count_ratio)/len(count_ratio)*100}% всех токенов')

  elif model_type == 'my_gensim':
    for elem in tqdm(X_train_clear):
      X_train_vectorized.append(text_to_vector_gensim(elem, vectorizer))

  print('Начинаю обучение классификатора')
  clf.fit(X_train_vectorized, y)



def eval_log_reg(X, y, vectorizer, clf, model_type):
  print('Предобработка и нормализация для валидации')
  X_train_clear = preprocess_corpus(X)

  X_val_vectorized = []
  print('Векторизация текста')

  if model_type == 'navec':
    for elem in tqdm(X_train_clear):
      X_val_vectorized.append(text_to_vector_navec(elem, vectorizer))

  elif model_type == 'rusvectors':
    count_ratio = []
    for elem in tqdm(X_train_clear):
      result = text_to_vector_rusvectores(elem, vectorizer)
      X_val_vectorized.append(result[0])
      count_ratio.append(result[1])

    print(f'Удалось найти {sum(count_ratio)/len(count_ratio)*100}% всех токенов')

  elif model_type == 'my_gensim':
    for elem in tqdm(X_train_clear):
      X_val_vectorized.append(text_to_vector_gensim(elem, vectorizer))


  y_pred = clf.predict(X_val_vectorized)

  print("Classification Report:")
  print(classification_report(y, y_pred, digits=4))


Проверим все w2v модели при одинаковых гиперпораметрах лог рега и оценим качество на валидационном сете

In [23]:
classifier_my_gensim = LogisticRegression(
      solver="saga",
      max_iter=1000,
      n_jobs=-1,
      class_weight="balanced",
      C=2.0
  )

fit_log_reg(X_train, y_train, model, classifier_my_gensim, 'my_gensim')

Предобработка и нормализация для обучения


100%|██████████| 59988/59988 [03:35<00:00, 277.81it/s]


Векторизация текста


100%|██████████| 59988/59988 [00:33<00:00, 1772.82it/s]


Начинаю обучение классификатора


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [14]:
eval_log_reg(X_val, y_val, model, classifier_my_gensim, 'my_gensim')

Предобработка и нормализация для валидации


100%|██████████| 19997/19997 [01:02<00:00, 319.59it/s]


Векторизация текста


100%|██████████| 19997/19997 [00:11<00:00, 1725.16it/s]


Classification Report:
                   precision    recall  f1-score   support

   69-я параллель     0.5890    0.8528    0.6967       163
           Бизнес     0.4182    0.7513    0.5373       398
      Бывший СССР     0.8256    0.8517    0.8385      1362
              Дом     0.7718    0.8578    0.8125       682
         Из жизни     0.6872    0.8063    0.7420       981
   Интернет и СМИ     0.7913    0.7707    0.7809      1387
             Крым     0.4400    0.8333    0.5759       132
    Культпросвет      0.1453    0.8525    0.2482        61
         Культура     0.8988    0.7287    0.8049      1316
              Мир     0.8518    0.8031    0.8267      2884
  Наука и техника     0.8515    0.8787    0.8649      1129
      Путешествия     0.7391    0.8434    0.7878       645
           Россия     0.8375    0.6449    0.7287      3030
Силовые структуры     0.6818    0.7718    0.7240      1385
            Спорт     0.9664    0.9597    0.9630      2009
         Ценности     0.8990    

In [14]:
classifier_navec = LogisticRegression(
      solver="saga",
      max_iter=1000,
      n_jobs=-1,
      class_weight="balanced",
      C=2.0
  )

fit_log_reg(X_train, y_train, navec, classifier_navec, 'navec')

Предобработка и нормализация для обучения


100%|██████████| 59988/59988 [03:00<00:00, 332.33it/s]


Векторизация текста


100%|██████████| 59988/59988 [01:39<00:00, 600.57it/s]


Начинаю обучение классификатора


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [57]:
eval_log_reg(X_val, y_val, navec, classifier_navec, 'navec')

Предобработка и нормализация для валидации


100%|██████████| 19997/19997 [01:10<00:00, 283.28it/s]


Векторизация текста


100%|██████████| 19997/19997 [00:42<00:00, 470.07it/s]


Classification Report:
                   precision    recall  f1-score   support

   69-я параллель     0.8889    0.4908    0.6324       163
           Бизнес     0.3740    0.7085    0.4896       398
      Бывший СССР     0.8113    0.8715    0.8404      1362
              Дом     0.7292    0.8372    0.7795       682
         Из жизни     0.6533    0.7798    0.7110       981
   Интернет и СМИ     0.7372    0.7686    0.7526      1387
             Крым     0.3539    0.8258    0.4955       132
    Культпросвет      0.1231    0.9344    0.2176        61
         Культура     0.9188    0.6360    0.7517      1316
              Мир     0.8527    0.7770    0.8131      2884
  Наука и техника     0.8290    0.8671    0.8476      1129
      Путешествия     0.6602    0.8403    0.7394       645
           Россия     0.8547    0.5881    0.6968      3030
Силовые структуры     0.6130    0.7913    0.6908      1385
            Спорт     0.9647    0.9522    0.9584      2009
         Ценности     0.8603    

In [ ]:
classifier_rusvectors = LogisticRegression(
      solver="saga",
      max_iter=1000,
      n_jobs=-1,
      class_weight="balanced",
      C=2.0
  )

fit_log_reg(X_train, y_train, rusvectors, classifier_rusvectors, 'rusvectors')

Предобработка и нормализация для обучения


100%|██████████| 59988/59988 [02:56<00:00, 339.13it/s]


Векторизация текста


100%|██████████| 59988/59988 [00:52<00:00, 1139.82it/s]


Удалось найти 63.86136335396178% всех токенов
Начинаю обучение классификатора


In [65]:
eval_log_reg(X_val, y_val, rusvectors, classifier_rusvectors, 'rusvectors')

Предобработка и нормализация для валидации


100%|██████████| 19997/19997 [01:06<00:00, 302.30it/s]


Векторизация текста


100%|██████████| 19997/19997 [00:20<00:00, 983.78it/s]


Удалось найти 63.86534689740907% всех токенов
Classification Report:
                   precision    recall  f1-score   support

   69-я параллель     0.2593    0.7730    0.3883       163
           Бизнес     0.3456    0.6608    0.4538       398
      Бывший СССР     0.7599    0.7878    0.7736      1362
              Дом     0.6401    0.7669    0.6978       682
         Из жизни     0.6106    0.7513    0.6737       981
   Интернет и СМИ     0.6940    0.6640    0.6787      1387
             Крым     0.4624    0.6061    0.5246       132
    Культпросвет      0.1145    0.8525    0.2019        61
         Культура     0.7867    0.7371    0.7611      1316
              Мир     0.7999    0.7292    0.7629      2884
  Наука и техника     0.7626    0.8423    0.8005      1129
      Путешествия     0.6014    0.7767    0.6779       645
           Россия     0.8448    0.4436    0.5817      3030
Силовые структуры     0.5965    0.7300    0.6565      1385
            Спорт     0.9524    0.9263    0.9

In [17]:
tfidf_vectorizer = TfidfVectorizer(
    preprocessor=lambda x: preprocess_one_cached(x, tfidf=True),
    token_pattern=r"(?u)\b[\wёЁ]+\b",
    ngram_range=(1, 1),
    min_df=3,
    max_features=100_000,
    sublinear_tf=True,
    dtype=np.float32
)

tfidf_vectorizer.fit(X_train)

TfidfVectorizer(dtype=<class 'numpy.float32'>, max_features=100000, min_df=3,
                preprocessor=<function <lambda> at 0x79869f68ed40>,
                sublinear_tf=True, token_pattern='(?u)\\b[\\wёЁ]+\\b')

In [15]:
def get_tfidf_weighted_embeddings(texts, tfidf_vectorizer, w2v_model):
    texts_clear = [preprocess_one_cached(t, tfidf=True) for t in texts]

    tfidf_matrix = tfidf_vectorizer.transform(texts_clear)
    feature_names = tfidf_vectorizer.get_feature_names_out()

    word_embeddings = []
    valid_indices = []

    for i, word in enumerate(feature_names):
        if word in w2v_model.wv:
            word_embeddings.append(w2v_model.wv[word])
            valid_indices.append(i)

    if not word_embeddings:
        raise ValueError("Нет слов, общих между TF-IDF словарем и моделью Navec")

    emb_matrix = np.array(word_embeddings, dtype=np.float32)
    tfidf_filtered = tfidf_matrix[:, valid_indices]
    weighted_embeddings = tfidf_filtered.dot(emb_matrix)

    return weighted_embeddings

Возьмем гиперпараметры для лог рега из ДЗ№1 подобранные в ходе рандомного поиска с кросс валидацией

In [20]:
%%time

classifier_my_gensim_tfidf = LogisticRegression(
      solver="saga",
      max_iter=1000,
      n_jobs=-1,
      class_weight="balanced",
      C=2.0
  )

classifier_my_gensim_tfidf.fit(get_tfidf_weighted_embeddings(X_train, tfidf_vectorizer, model), y_train)

CPU times: user 26min 57s, sys: 2.22 s, total: 26min 59s
Wall time: 27min 11s


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


LogisticRegression(C=2.0, class_weight='balanced', max_iter=1000, n_jobs=-1,
                   solver='saga')

In [21]:
y_pred = classifier_my_gensim_tfidf.predict(get_tfidf_weighted_embeddings(X_val, tfidf_vectorizer, model))

print("Classification Report:")
print(classification_report(y_val, y_pred, digits=4))

Classification Report:
                   precision    recall  f1-score   support

   69-я параллель     0.6436    0.7975    0.7123       163
           Бизнес     0.4322    0.7286    0.5426       398
      Бывший СССР     0.8213    0.8568    0.8387      1362
              Дом     0.7257    0.8416    0.7794       682
         Из жизни     0.6937    0.7941    0.7405       981
   Интернет и СМИ     0.7813    0.7650    0.7730      1387
             Крым     0.4612    0.7197    0.5621       132
    Культпросвет      0.2350    0.7049    0.3525        61
         Культура     0.8581    0.8040    0.8301      1316
              Мир     0.8530    0.7965    0.8237      2884
  Наука и техника     0.8454    0.8769    0.8609      1129
      Путешествия     0.7224    0.8310    0.7729       645
           Россия     0.8238    0.6528    0.7284      3030
Силовые структуры     0.6724    0.7560    0.7118      1385
            Спорт     0.9634    0.9577    0.9606      2009
         Ценности     0.8855    

Сравним все на тестовой выборке

In [22]:
y_pred = classifier_my_gensim_tfidf.predict(get_tfidf_weighted_embeddings(X_test, tfidf_vectorizer, model))

print("Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

Classification Report:
                   precision    recall  f1-score   support

   69-я параллель     0.6684    0.7914    0.7247       163
           Бизнес     0.3973    0.6742    0.5000       399
      Бывший СССР     0.8133    0.8443    0.8285      1362
              Дом     0.7315    0.8827    0.8000       682
         Из жизни     0.6863    0.7737    0.7274       981
   Интернет и СМИ     0.7545    0.7578    0.7561      1387
             Крым     0.4633    0.7652    0.5771       132
    Культпросвет      0.2652    0.7869    0.3967        61
         Культура     0.8582    0.8138    0.8354      1316
              Мир     0.8624    0.7951    0.8273      2884
  Наука и техника     0.8366    0.8618    0.8490      1129
      Путешествия     0.7380    0.8558    0.7925       645
           Россия     0.8272    0.6462    0.7256      3030
Силовые структуры     0.6701    0.7451    0.7056      1385
            Спорт     0.9599    0.9542    0.9571      2009
         Ценности     0.8836    

In [25]:
eval_log_reg(X_test, y_test , model, classifier_my_gensim, 'my_gensim')

Предобработка и нормализация для валидации


100%|██████████| 19997/19997 [01:32<00:00, 216.87it/s]


Векторизация текста


100%|██████████| 19997/19997 [00:10<00:00, 1983.77it/s]


Classification Report:
                   precision    recall  f1-score   support

   69-я параллель     0.6037    0.8037    0.6895       163
           Бизнес     0.4208    0.6992    0.5254       399
      Бывший СССР     0.8222    0.8554    0.8384      1362
              Дом     0.7738    0.8475    0.8090       682
         Из жизни     0.6664    0.8002    0.7272       981
   Интернет и СМИ     0.7350    0.7621    0.7483      1387
             Крым     0.5268    0.8939    0.6629       132
    Культпросвет      0.0537    0.9836    0.1019        61
         Культура     0.9708    0.3792    0.5454      1316
              Мир     0.8613    0.8013    0.8302      2884
  Наука и техника     0.8619    0.8680    0.8650      1129
      Путешествия     0.7698    0.8450    0.8056       645
           Россия     0.8360    0.6343    0.7213      3030
Силовые структуры     0.6883    0.7430    0.7146      1385
            Спорт     0.9637    0.9512    0.9574      2009
         Ценности     0.8978    

In [15]:
eval_log_reg(X_test, y_test, navec, classifier_navec, 'navec')

Предобработка и нормализация для валидации


100%|██████████| 19997/19997 [00:57<00:00, 347.11it/s]


Векторизация текста


100%|██████████| 19997/19997 [00:34<00:00, 576.83it/s]


Classification Report:
                   precision    recall  f1-score   support

   69-я параллель     0.5124    0.7607    0.6123       163
           Бизнес     0.4193    0.6642    0.5141       399
      Бывший СССР     0.8049    0.8421    0.8231      1362
              Дом     0.7411    0.8563    0.7946       682
         Из жизни     0.6927    0.7421    0.7165       981
   Интернет и СМИ     0.7390    0.7246    0.7317      1387
             Крым     0.2707    0.9394    0.4203       132
    Культпросвет      0.2168    0.8033    0.3415        61
         Культура     0.8252    0.7675    0.7953      1316
              Мир     0.8385    0.7923    0.8148      2884
  Наука и техника     0.8335    0.8645    0.8487      1129
      Путешествия     0.8072    0.7333    0.7685       645
           Россия     0.8326    0.5924    0.6922      3030
Силовые структуры     0.6185    0.7726    0.6870      1385
            Спорт     0.9628    0.9418    0.9522      2009
         Ценности     0.8683    

In [18]:
eval_log_reg(X_test, y_test, rusvectors, classifier_rusvectors, 'rusvectors')

Предобработка и нормализация для валидации


100%|██████████| 19997/19997 [01:09<00:00, 286.13it/s]


Векторизация текста


100%|██████████| 19997/19997 [00:21<00:00, 916.97it/s] 


Удалось найти 63.88470601306% всех токенов
Classification Report:
                   precision    recall  f1-score   support

   69-я параллель     0.2723    0.7853    0.4044       163
           Бизнес     0.3471    0.6516    0.4530       399
      Бывший СССР     0.7623    0.7863    0.7741      1362
              Дом     0.6739    0.7786    0.7224       682
         Из жизни     0.5977    0.7543    0.6670       981
   Интернет и СМИ     0.6717    0.6770    0.6743      1387
             Крым     0.2840    0.9015    0.4319       132
    Культпросвет      0.4024    0.5410    0.4615        61
         Культура     0.8307    0.7272    0.7755      1316
              Мир     0.8066    0.7375    0.7705      2884
  Наука и техника     0.7998    0.8175    0.8086      1129
      Путешествия     0.6219    0.7829    0.6932       645
           Россия     0.7864    0.4957    0.6081      3030
Силовые структуры     0.5860    0.6888    0.6333      1385
            Спорт     0.9520    0.9288    0.9403